In [8]:
import numpy as np
import sys
sys.path.append('..')
import os
import torch
from torch.utils.data import TensorDataset, DataLoader
import warnings
import pickle
warnings.filterwarnings("ignore")
from collections import OrderedDict
import xarray as xr

from src.data_assemble.assemble_ml import *
from src.data_assemble.assemble_conv import *
from src.models.utils import *
from src.data_utils.data_processing import *

In [9]:
path_to_files = '../data/stash/WindProject/cmip_stash/*.nc'
filter_dict = {"years": ['2016'], "bands": ['Wind_', 'pr_', 'tasmax', 'tasmin']}
# rectangle_coords = {'lat_min': 43.38, 'lat_max': 51.52,'lon_min': 35.12, 'lon_max': 44.45}  # KK + Rostov + Belgorod
rectangle_coords = {'lat_min': 43.5, 'lat_max': 48,'lon_min': 36.5, 'lon_max': 42} # Krasnodarskiy Krai was 41.75
# rectangle_coords = {'lat_min': 43.38, 'lat_max': 50.23,'lon_min': 36.58, 'lon_max': 44.45}  # KK + Rostov
# rectangle_coords = {'lat_min': 24, 'lat_max': 31,'lon_min': 272, 'lon_max': 280} # Florida
target_res = {'lon_res': 0.25, 'lat_res': 0.25}

blocks = make_blocks([path_to_files], filter_dict, rectangle_coords, target_res, half_side_size=3)

100%|██████████| 4/4 [00:01<00:00,  3.41it/s]


In [4]:
# df = pd.read_csv('../data/stash/WindProject/weather_stations/data_meteo_full.csv')
# weatherstation_list = pd.read_csv('../weatherstation_list.csv')
# start = '2022-09-01'
# end = '2022-10-31'
# station_names = ['Богородицкое-Фенино', 'Готня', 'Валуйки', 'Чертково', 'Цимлянск(Волгодонск)', 'Таганрог',  
#                  'Гигант', 'Ремонтное', 'Приморско-Ахтарск', 'Краснодар, Круглик', 'Анапа', 'Туапсе', 'Армавир', 'Сочи',
#                  'Красная Поляна']
# stations_pixs = get_pixel_stations(path_to_files, filter_dict, station_names, weatherstation_list, rectangle_coords, target_res)
# target = get_y(df, start, end, station_names, speed_th=20)

100%|██████████| 3650/3650 [00:00<00:00, 9831.62it/s] 


In [10]:
X = assemble_numpy_ds(blocks=blocks, target='', stations_pixs='', include_target=False)

In [11]:
# WRITE PATH TO TERRABYTE
path_to_dump = os.path.join('..', 'data', 'stash', 'WindProject', 'nn_data_grid_inference_kk_extended')
obj_path = os.path.join(path_to_dump, 'objects')
for k in X.keys():
    X_station = X[k]

    st_path = os.path.join(path_to_dump, str(k))
    if not os.path.isdir(st_path):
        os.makedirs(st_path)
# WRITE PATH TO TERRABYTE    
    with open(os.path.join(st_path, 'objects.npy'),'wb') as f:
        pickle.dump(X_station, f)
        # np.save(f, X_station)